# TFVI-DPGMM revision: 5 seeds

In [ ]:
import pandas as pd
import torch
import matplotlib
import matplotlib.pyplot as plt
from abc import abstractmethod
from typing import List, Callable, Union, Any, TypeVar, Tuple
from itertools import cycle
Tensor = TypeVar('torch.tensor')

import numpy as np
import time
from Uncertainty_calibration_scores import *

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.optimize import linear_sum_assignment

from Truncation_free_VI import onlineVDP_suff_stat
from Truncation_free_VI import vdpgm
from sklearn.cluster import KMeans
from collections import Counter
from copy import deepcopy

experiment_seeds = [42, 1, 2, 3, 6]

# Dataset

In [ ]:
'''use a numerical dataset to test the model
label  damaged_floor     damage_extent         number_of_samples
0           0            0%                     1500
1           1            5%                     500
2           1            10%                    500
3           2,4          10%, 10%               500
4           1,3,5        10%,15%,20%            500
5           2,4,6        15%,20%,25%            500
6           1,3,5,7      10%,15%,20%,25%        500
7           1,2,4,6,8    10%,15%,20%,25%,30%    500

'''
# Read data from CSV
features = pd.read_csv("TF_mag_numerical_8class_5000samples_20dB.csv")
features = features.astype("float32")
# # # Convert DataFrame to PyTorch tensors
X = torch.tensor(features.values[:7500,:])
X1 = X[1000:1500,:]
X2 = X[1800:2300,:]
X = torch.cat([X1, X2], dim=0)

X = X.t()

input_dim = X.shape[1]
print(X.shape)


a = []
for i in range(1500):
    a.append(0)
for i in np.arange(1500,2000):
    a.append(1)
for i in np.arange(2000,2500):
    a.append(2)
for i in np.arange(2500,3000):
    a.append(3)
for i in np.arange(3000,3500):
    a.append(4)
for i in np.arange(3500,4000):
    a.append(5)
for i in np.arange(4000,4500):
    a.append(6)
for i in np.arange(4500,5000):
    a.append(7)
print(len(a))

y0 = torch.tensor(a)
y0 = y0.unsqueeze(1)
y = [int(_) for _ in y0] 
y = torch.tensor(y)
num_classes = y.max().item() + 1
print(f"number of total classes: {num_classes}")

plt.plot(y)
plt.show()

# Helper functions

In [ ]:
def move_dict_to_device(state_dict, device):
    '''move all tensors in state_dict to the device'''
    new_state_dict = {}
    for k, v in state_dict.items():
        if isinstance(v, torch.Tensor):
            new_state_dict[k] = v.to(device)
        else:
            new_state_dict[k] = v
    return new_state_dict

# Five-seed experiment

In [ ]:
def unsupervised_clustering_accuracy(y: Union[np.ndarray, torch.Tensor], y_pred: Union[np.ndarray, torch.Tensor]) -> tuple:
        """Unsupervised Clustering Accuracy
        """
        assert len(y_pred) == len(y)
        u = np.unique(y)
        n_true_clusters = len(u)
        v = np.unique(y_pred)
        n_pred_clusters = len(v)
        map_u = dict(zip(u, range(n_true_clusters)))
        map_v = dict(zip(v, range(n_pred_clusters)))
        inv_map_u = {v: k for k, v in map_u.items()}
        inv_map_v = {v: k for k, v in map_v.items()}
        r = np.zeros((n_pred_clusters, n_true_clusters), dtype=np.int64)
        for y_pred_, y_ in zip(y_pred, y):
            if y_ in map_u:
                r[map_v[y_pred_], map_u[y_]] += 1
        reward_matrix  = np.concatenate((r, r, r), axis=1)
        cost_matrix = reward_matrix.max() - reward_matrix
        row_assign, col_assign = linear_sum_assignment(cost_matrix)

        # Construct optimal assignments matrix
        row_assign = row_assign.reshape((-1, 1))  # (n,) to (n, 1) reshape
        col_assign = col_assign.reshape((-1, 1))  # (n,) to (n, 1) reshape
        assignments = np.concatenate((row_assign, col_assign), axis=1)
        assignments = [[inv_map_v[x], inv_map_u[y%n_true_clusters]] for x, y in assignments]

        optimal_reward = reward_matrix[row_assign, col_assign].sum() * 1.0
        return optimal_reward / y_pred.size, assignments  

def damage_detection_accuracy(hard_assign, healthy_count=1500, healthy_reference_fraction=0.8, min_count=11):
    health_labels_pre = np.unique(hard_assign[:int(healthy_count * healthy_reference_fraction)])
    counts = Counter(hard_assign[:int(healthy_count * healthy_reference_fraction)])
    health_labels = [label for label in health_labels_pre if counts[label] >= min_count]

    fn, fp = 0, 0
    for i in range(len(hard_assign)):
        if i <= healthy_count and hard_assign[i] not in health_labels:
            fp += 1
        elif i > healthy_count and hard_assign[i] in health_labels:
            fn += 1

    dda = 1 - (fp + fn) / len(hard_assign)
    return dda, fp, fn, health_labels

def run_tfvi_dpgmm_once(seed, X, y, initialize_with_kmeans=True):
    torch.manual_seed(seed)
    np.random.seed(seed)

    pca_components = 10
    samples_per_segment = 50

    pca = PCA(n_components=pca_components)
    data_pca = pca.fit_transform(X).astype("float32")
    data_pca = data_pca.T
    data_pca = torch.tensor(data_pca)
    print('PCA data shape:', data_pca.shape)

    dpmm_param = dict(
        alpha=torch.tensor(1.0),
        beta0=torch.tensor(0.01),
        max_iter_merge=100,
        max_iter_split=100,
        max_merge=10,
        max_split=10,
        ite=300,
        do_merge=0,
    )

    opts = onlineVDP_suff_stat.mkopts_vdp(**dpmm_param)
    prior = onlineVDP_suff_stat.mk_hp_prior(data_pca, **dpmm_param)

    if initialize_with_kmeans:
        kmeans_components = 50
        kmeans = KMeans(n_clusters=kmeans_components, random_state=seed)
        z_init = kmeans.fit_predict(data_pca.numpy().T)
        q_of_z = torch.zeros(data_pca.shape[1], kmeans_components)
        for i in range(data_pca.shape[1]):
            q_of_z[i, z_init[i]] = 1.0
        sum_stat = onlineVDP_suff_stat.compute_summary_stats(data_pca, q_of_z, opts)
        hp_posterior = onlineVDP_suff_stat.mk_hp_posterior(prior, sum_stat, opts)
        current_result = {'sum_stat': sum_stat, 'q_of_z': q_of_z}
    else:
        current_result = {}

    start_time = time.time()
    current_result = onlineVDP_suff_stat.online_vdpgmm_suffstat(
        data_pca,
        prior,
        opts,
        move_dict_to_device(current_result, data_pca.device),
    )
    runtime = time.time() - start_time
    print(f"VI-DPGMM took {runtime} seconds.")

    q_of_z = current_result['q_of_z']
    print(q_of_z.shape)

    resp_tensor = q_of_z[:, :]
    hard_assign_tensor = resp_tensor.argmax(axis=1)

    resp = resp_tensor.detach().cpu().numpy()
    hard_assign = hard_assign_tensor.detach().cpu().numpy().astype(int)
    y_true = y.numpy().astype(int)

    acc, assignments = unsupervised_clustering_accuracy(y_true, hard_assign.astype(int))
    dda, fp, fn, health_labels = damage_detection_accuracy(hard_assign)

    BS, BSc, BSS, P_hat, R, Q, classes = dpgmm_brier_from_responsibilities(
        R=resp,
        y_true=y_true,
        labeled_mask=None,
        classes=None,
        sample_weight=None,
    )

    P_dpg, _ = probs_with_cv_Q(
        R=resp,
        y_true=y_true,
        classes=classes,
        kfold=5,
    )
    ece_dpg_top = ece_toplabel(P_dpg, y_true, n_bins=15)
    ece_dpg_ovr = ece_ovr(P_dpg, y_true=y_true, n_bins=15, classes=classes)
    ece_dpg_ovr_bal = ece_ovr_classbalanced(P_dpg, y_true, n_bins=15)

    return {
        'seed': seed,
        'accuracy': acc,
        'damage_detection_accuracy': dda,
        'false_positive': fp,
        'false_negative': fn,
        'brier_score': BS,
        'brier_skill_score': BSS,
        'ece_toplabel': ece_dpg_top,
        'ece_ovr': ece_dpg_ovr,
        'ece_balanced': ece_dpg_ovr_bal,
        'runtime_seconds': runtime,
        'num_components': len(np.unique(hard_assign)),
        'assignments': assignments,
        'healthy_labels': health_labels,
        'predicted_clusters': hard_assign.copy(),
        'y_true': y_true.copy(),
    }

In [ ]:
seed_results = []

for run_id, seed in enumerate(experiment_seeds, start=1):
    print(f"\n===== Seed {seed} ({run_id}/{len(experiment_seeds)}) =====")
    result = run_tfvi_dpgmm_once(seed=seed, X=X, y=y, initialize_with_kmeans=True)
    seed_results.append(result)

    print(
        f"Seed {seed}: ACC={result['accuracy']:.6f}, "
        f"DDA={result['damage_detection_accuracy']:.6f}, "
        f"Brier={result['brier_score']:.6f}, "
        f"BSS={result['brier_skill_score']:.6f}, "
        f"ECE_top={result['ece_toplabel']:.6f}, "
        f"runtime={result['runtime_seconds']:.2f}s, "
        f"K={result['num_components']}"
    )

results_df = pd.DataFrame(seed_results)

metric_columns = [
    'accuracy',
    'damage_detection_accuracy',
    'brier_score',
    'ece_balanced',
    'runtime_seconds',
]

summary_df = pd.DataFrame({
    'mean': results_df[metric_columns].mean(),
    'std': results_df[metric_columns].std(ddof=1),
})

In [ ]:
print('\nPer-seed results')
display(results_df[[
    'seed',
    'accuracy',
    'damage_detection_accuracy',
    'false_positive',
    'false_negative',
    'brier_score',
    'ece_balanced',
    'runtime_seconds',
    'num_components',
]])

print('\nMean and sample standard deviation over seeds')
display(summary_df)